# Hyperparameter tuning for ensemble models

In [28]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data, target = fetch_california_housing(return_X_y=True, as_frame=True)
target *= 100  # rescale the target in k$
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=0
)

## Random Forest
- main parameter = `n_estimators`
- fix `n_estimators = 100` (default value)
- more tree => better generalization, but more tree => slow fitting and predictio time
- tune the hyperparameter max_features, which controls the size of the random subset of features to consider when looking for the best split when growing the trees: smaller values for max_features lead to more random trees with hopefully more uncorrelated prediction errors. However if max_features is too small, predictions can be too random, even after averaging with the trees in the ensemble.
- `max_depth` and `max_leaf_nodes`

In [29]:
print(f"In this case, n_features={len(data.columns)}")

In this case, n_features=8


In [30]:
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

param_distributions = {
    "max_features": [1, 2, 3, 5, None],
    "max_leaf_nodes": [10, 100, 1000, None],
    "min_samples_leaf": [1, 2, 5, 10, 20, 50, 100],
}
search_cv = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=2),
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_error",
    n_iter=10,
    random_state=0,
    n_jobs=2,
)
search_cv.fit(data_train, target_train)

columns = [f"param_{name}" for name in param_distributions.keys()]
columns += ["mean_test_error", "std_test_error"]
cv_results = pd.DataFrame(search_cv.cv_results_)
cv_results["mean_test_error"] = -cv_results["mean_test_score"]
cv_results["std_test_error"] = cv_results["std_test_score"]
cv_results[columns].sort_values(by="mean_test_error")

,param_max_features,param_max_leaf_nodes,param_min_samples_leaf,mean_test_error,std_test_error
3,2,None,2,33.793108,0.684197
0,2,1000,10,37.009619,0.536735
7,None,None,20,37.360315,0.486945
4,5,100,2,40.072588,0.449727
8,None,100,10,40.413725,0.714505
6,None,1000,50,40.805294,0.395275
2,1,100,1,50.127721,0.780255
9,1,100,2,50.220753,0.923140
5,1,None,100,54.630284,0.689114
1,3,10,10,54.824381,0.792111


In [31]:
error = -search_cv.score(data_test, target_test)
print(
    f"On average, our random forest regressor makes an error of {error:.2f} k$"
)

On average, our random forest regressor makes an error of 33.80 k$


## Histogram gradient-boosting decision trees
- The important hyperparameters are `max_iter`, `learning_rate`, and `max_depth` or `max_leaf_nodes`
- `max_iter` : controls the number of trees in the estimator
- `max_depth` or `max_leaf_nodes` : controls depth of the trees
- the tree used in gradient-boosting should have a low depth, typically between 3 to 8 levels, or few leaves ( 23=8
  to  28=256). Having very weak learners at each step helps reducing overfitting.
- the deeper the trees, the faster the residuals are corrected and then less learners are required. 
- Therefore, it can be beneficial to increase max_iter if max_depth is low.
- `learning_rate` : When fitting the residuals, we would like the tree to try to correct all possible errors or only a fraction of them. The learning-rate allows you to control this behaviour. A small learning-rate value would only correct the residuals of very few samples. If a large learning-rate is set (e.g., 1), we would fit the residuals of all samples. So, with a very low learning-rate, we would need more estimators to correct the overall error. However, a too large learning-rate tends to obtain an overfitted ensemble, similar to having very deep trees.

In [32]:
from scipy.stats import loguniform
from sklearn.ensemble import HistGradientBoostingRegressor

param_distributions = {
    "max_iter": [3, 10, 30, 100, 300, 1000],
    "max_leaf_nodes": [2, 5, 10, 20, 50, 100],
    "learning_rate": loguniform(0.01, 1),
}
search_cv = RandomizedSearchCV(
    HistGradientBoostingRegressor(),
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_error",
    n_iter=20,
    random_state=0,
    n_jobs=2,
)
search_cv.fit(data_train, target_train)

columns = [f"param_{name}" for name in param_distributions.keys()]
columns += ["mean_test_error", "std_test_error"]
cv_results = pd.DataFrame(search_cv.cv_results_)
cv_results["mean_test_error"] = -cv_results["mean_test_score"]
cv_results["std_test_error"] = cv_results["std_test_score"]
cv_results[columns].sort_values(by="mean_test_error")

,param_max_iter,param_max_leaf_nodes,param_learning_rate,mean_test_error,std_test_error
14,300,100,0.018640,31.027583,0.227997
6,300,20,0.047293,32.018723,0.308458
2,30,50,0.176656,32.588693,0.253512
13,300,10,0.297739,32.993833,0.307791
9,100,20,0.083745,33.065636,0.462947
19,100,10,0.215543,33.372219,0.488499
12,100,20,0.067503,33.621295,0.418671
16,300,5,0.059290,35.822362,0.402681
1,100,5,0.160519,36.241319,0.518536
0,1000,2,0.125207,40.810068,0.547629


In [33]:
error = -search_cv.score(data_test, target_test)
print(f"On average, our HGBT regressor makes an error of {error:.2f} k$")

On average, our HGBT regressor makes an error of 30.37 k$


| **Bagging & Random Forests**                     | **Boosting**                                        |
|--------------------------------------------------|-----------------------------------------------------|
| fit trees **independently**                      | fit trees **sequentially**                          |
| each **deep tree overfits**                      | each **shallow tree underfits**                     |
| averaging the tree predictions **reduces overfitting** | sequentially adding trees **reduces underfitting** |
| generalization improves with the number of trees | too many trees may cause overfitting                |
| does not have a `learning_rate` parameter        | fitting the residuals is controlled by the `learning_rate` |